In [29]:
import os

from typing import NotRequired

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import PostgresStore
from rich import print as rprint

load_dotenv(override=True)

# Long-term Memory

## put()/get()

### Base on InMemoryStore

In [1]:
store = InMemoryStore()

namespace = ("users", "user-01")
key = "profile"
value = {"name": "V!p3N"}

store.put(namespace, key, value)

item = store.get(namespace, key)

rprint(item)

Item(namespace=['users', 'user-01'], key='profile', value={'name': 'V!p3N'}, 
created_at='2026-08-02T12:46:37.449236+00:00', updated_at='2026-08-02T12:46:37.449241+00:00')

In [2]:
value = {"name": "V!p3N[doge]"}

store.put(namespace, key, value)

item = store.get(namespace, key)

rprint(item)

Item(namespace=['users', 'user-01'], key='profile', value={'name': 'V!p3N[doge]'}, 
created_at='2026-08-02T12:47:22.372843+00:00', updated_at='2026-08-02T12:47:22.372847+00:00')

### Base on PostgresStore

In [6]:
DB_URL = os.getenv("LANGCHAIN_POSTGRES_URL")
if not DB_URL:
    raise RuntimeError("请在 .env 中设置 LANGCHAIN_POSTGRES_URL")
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    store.put(namespace, key, value)
    item = store.get(namespace, key)
    rprint(item)

Item(namespace=['users', 'user-01'], key='profile', value={'name': 'V!p3N[doge]'}, 
created_at='2026-08-02T12:50:29.015535+00:00', updated_at='2026-08-02T12:50:29.015535+00:00')

In [7]:
value = {"name": "V!p3N"}
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    store.put(namespace, key, value)
    item = store.get(namespace, key)
    rprint(item)

Item(namespace=['users', 'user-01'], key='profile', value={'name': 'V!p3N'}, 
created_at='2026-08-02T12:50:29.015535+00:00', updated_at='2026-08-02T13:32:43.453440+00:00')

## search()

### 按照namespace前缀搜索

In [8]:
store = InMemoryStore()

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [9]:
for item in store.search(("users",)):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T14:22:06.349389+00:00', updated_at='2026-08-02T14:22:06.349394+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-08-02T14:22:06.349533+00:00', updated_at='2026-08-02T14:22:06.349534+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T14:22:06.351652+00:00', updated_at='2026-08-02T14:22:06.351656+00:00', score=None)


In [10]:
for item in store.search(("users", "Alice")):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T14:22:06.349389+00:00', updated_at='2026-08-02T14:22:06.349394+00:00', score=None)


### 按照filter过滤

In [14]:
for item in store.search(("users",), filter={"course": "数字电路与模拟电路", "sports": "羽毛球", }):
    print(item)

Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T14:22:06.351652+00:00', updated_at='2026-08-02T14:22:06.351656+00:00', score=None)


### 按照语义搜索

In [15]:
from langgraph.store.memory import InMemoryStore


# 自定义嵌入函数
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]


index_config = {
    "embed": embed,
    "dims": 6,
    "fields": ["$", "course"]
}
store = InMemoryStore(
    index=index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [16]:
from pprint import pprint

pprint(store._vectors)

defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x1207136a0>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {'preferences': {'$': [1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0],
                                                                          'course': [1.0,
                                                                                     1.0,
                                                                                     1.0,
                                                                  

In [17]:
from pprint import pprint

pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['$'])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


#### 使用嵌入模型

In [23]:
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings

import os

load_dotenv(override=True)

embedding_model = init_embeddings(
    model="openai:qwen/qwen3-embedding-4b",
    base_url=os.getenv("OPENROUTER_API_BASE"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
    dimensions=2048,
)

index_config = {
    "embed": embedding_model,
    "dims": 2048,
    "fields": ["$"]
}
store = InMemoryStore(
    index=index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [24]:
for item in store.search(("users",), query="数电模电"):
    print(item)

Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T15:30:54.576964+00:00', updated_at='2026-08-02T15:30:54.576968+00:00', score=0.6064299516625743)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-08-02T15:30:53.857327+00:00', updated_at='2026-08-02T15:30:53.857336+00:00', score=0.5823286858027482)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T15:30:53.187588+00:00', updated_at='2026-08-02T15:30:53.187605+00:00', score=0.47591384224607675)


In [26]:
for item in store.search(("users",), query="跑步", limit=2):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-02T15:30:53.187588+00:00', updated_at='2026-08-02T15:30:53.187605+00:00', score=0.5097573272191457)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-08-02T15:30:53.857327+00:00', updated_at='2026-08-02T15:30:53.857336+00:00', score=0.4978248181784529)


## Use long-term memory in Agent

### Tool

#### Base on InMemoryStore

In [31]:
store = InMemoryStore()


class CustomState(AgentState):
    user_id: NotRequired[str]


@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"


@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"


model = init_chat_model("deepseek:deepseek-v4-flash")
agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store=store,
    system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
    state_schema=CustomState,
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_00_Wa3zZ0BVamzGDEdfnjip9272)
 Call ID: call_00_Wa3zZ0BVamzGDEdfnjip9272
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好呀，小花！很高兴认识你 🌸 我已经把你的名字记住了。有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_0333FfBCepbDYJFWw49S5960)
 Call ID: call_00_0333FfBCepbDYJFWw49S5960
  Args:
==========================

#### Base on PostgresStore

In [32]:
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    agent = create_agent(
        model=model,
        tools=[save_user_info, get_user_info],
        store=store,
        system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
        state_schema=CustomState,
    )

    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": [HumanMessage("我是谁")],
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你 😊 让我先把你的名字记下来。
Tool Calls:
  save_user_info (call_00_nVolotIuUkezYRTMQBGp8941)
 Call ID: call_00_nVolotIuUkezYRTMQBGp8941
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

已经记住啦！以后我会记得你叫小花的 🌸

今天有什么我可以帮你的吗？无论是聊天、解答问题，还是帮忙处理一些事情，我都很乐意帮忙～
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_RA7MWohPxrKiQNXpzCka9909)
 Call ID: call_00_RA7MW